# <center> Лабораторна робота №11. Налаштування гіперпараметрів регресійних моделей для оцінювання якості вина


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, LassoCV, Lasso
from sklearn.ensemble import RandomForestRegressor

У завданні буде використано набір даних про якість білого вина(репозиторій UCI)
archive.ics.uci.edu/ml/machine-learning-databases/wine-quality.
Завантажте дані

In [2]:
from pathlib import Path
import urllib.request

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
data_path = data_dir / 'winequality-white.csv'
if not data_path.exists():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'
    urllib.request.urlretrieve(url, data_path)
data = pd.read_csv(data_path, sep=';')
print('Розмір набору даних:', data.shape)
display(data.sample(10, random_state=17))


Розмір набору даних: (4898, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
1682,7.2,0.25,0.28,14.4,0.055,55.0,205.0,0.99860,3.12,0.38,9.0,7
4181,6.6,0.25,0.32,5.6,0.039,15.0,68.0,0.99163,2.96,0.52,11.1,6
1992,7.0,0.12,0.28,6.3,0.057,17.0,103.0,0.99570,3.50,0.44,9.6,5
4239,5.7,0.28,0.36,1.8,0.041,38.0,90.0,0.99002,3.27,0.98,11.9,7
600,6.7,0.30,0.35,1.4,0.180,36.0,160.0,0.99370,3.11,0.54,9.4,6
2528,6.4,0.16,0.37,1.5,0.037,27.0,109.0,0.99345,3.38,0.50,9.8,6
1378,6.9,0.28,0.30,1.6,0.047,46.0,132.0,0.99180,3.35,0.38,11.1,7
548,6.5,0.18,0.31,1.7,0.044,30.0,127.0,0.99280,3.49,0.50,10.2,7
1662,6.7,0.21,0.49,1.4,0.047,30.0,114.0,0.99140,2.92,0.42,10.8,7
1370,7.6,0.28,0.39,1.9,0.052,23.0,116.0,0.99410,3.25,0.40,10.4,6


In [3]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


Відокремте цільову змінну, розділіть навчальну вибірку у відношенні 7:3 (30% - під задишену вибірку, нехай random_state=17) і нормалізуйте дані за допомогою StandartScaler


In [5]:
y = data['quality'].to_numpy()
feature_names = data.drop('quality', axis=1).columns
data.drop('quality', axis=1, inplace=True)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    data, y, test_size=0.3, random_state=17
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)


## Лінійна регресія


Навчіть просту лінійну модель регресії

In [6]:
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)


LinearRegression()

> **Питання 1 : Які середньоквадратичні помилки лінійної регресії на навчальній і відкоаденій вибірках ?**

In [7]:
print('Середньоквадратична помилка (навчальна вибірка): %.3f' % mean_squared_error(y_train, linreg.predict(X_train_scaled)))
print('Середньоквадратична помилка (відкладена вибірка): %.3f' % mean_squared_error(y_holdout, linreg.predict(X_holdout_scaled)))


Середньоквадратична помилка (навчальна вибірка): 0.558
Середньоквадратична помилка (відкладена вибірка): 0.584


Подивіться на коефіцієнти моделі і ранжуйте ознаки за впливом на якість вина (врахуйте, що великі за модулем негативні значення коефіцієнтів теж говорять про сильний вплив). Створіть для цього новий невеликий DataFrame.
> **Питання 2 : Яку ознаку лінійна регресія вважає найбільш впливовою на якість вина?**




In [8]:
linreg_coef = pd.DataFrame({'feature': feature_names, 'coefficient': linreg.coef_})
linreg_coef['absolute_influence'] = linreg_coef['coefficient'].abs()
linreg_coef = linreg_coef.sort_values('absolute_influence', ascending=False)
display(linreg_coef)
print('Найвпливовіша ознака:', linreg_coef.iloc[0]['feature'])


,feature,coefficient,absolute_influence
7,density,-0.665720,0.665720
3,residual sugar,0.538164,0.538164
1,volatile acidity,-0.192260,0.192260
8,pH,0.150036,0.150036
10,alcohol,0.129533,0.129533
0,fixed acidity,0.097822,0.097822
9,sulphates,0.062053,0.062053
5,free sulfur dioxide,0.042180,0.042180
6,total sulfur dioxide,0.014304,0.014304
4,chlorides,0.008127,0.008127


Найвпливовіша ознака: density


## Lasso-регресія

**Навчіть Lasso-регресію з невеликим коефіцієнтом alpha=0,01 (слабка регуляризація). Нехай знову random_state=17.**


In [9]:
lasso1 = Lasso(alpha=0.01, random_state=17, max_iter=10000)
lasso1.fit(X_train_scaled, y_train)


Lasso(alpha=0.01, max_iter=10000, random_state=17)

**Подивіться на коефіцієнти моделі і ранжуйте ознаки за впливом на якість вина. Яка ознака "відпала" першою, тобто найменш важлива для пояснення цільової змінної в моделі Lasso?**

In [10]:
lasso1_coef = pd.DataFrame({'feature': feature_names, 'coefficient': lasso1.coef_})
lasso1_coef['absolute_influence'] = lasso1_coef['coefficient'].abs()
display(lasso1_coef.sort_values('absolute_influence', ascending=False))
print('Найменш важлива ознака для Lasso:', lasso1_coef.loc[lasso1_coef['absolute_influence'].idxmin(), 'feature'])


,feature,coefficient,absolute_influence
10,alcohol,0.322425,0.322425
3,residual sugar,0.256363,0.256363
7,density,-0.235492,0.235492
1,volatile acidity,-0.188479,0.188479
8,pH,0.067277,0.067277
5,free sulfur dioxide,0.043088,0.043088
9,sulphates,0.029722,0.029722
4,chlorides,-0.002747,0.002747
0,fixed acidity,-0.000000,0.000000
2,citric acid,-0.000000,0.000000


Найменш важлива ознака для Lasso: fixed acidity


**Тепер визначте краще значення alpha в процесі 5-кратної крос-валідації. Використовуйте LassoCV і random_state = 17.**

In [11]:
alphas = np.logspace(-6, 2, 200)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=17, selection='random', max_iter=20000, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)
print('Оптимальне значення alpha:', lasso_cv.alpha_)


Оптимальне значення alpha: 0.0003107866187782014


In [12]:
lasso_cv.alpha_

np.float64(0.0003107866187782014)

Виведіть коефіцієнти "кращого" Lasso в порядку зменшення впливу на якість вина.
> **Питання 3: Яка ознака "занулилася першою" в налаштованій моделі LASSO?**

In [13]:
lasso_cv_coef = pd.DataFrame({'feature': feature_names, 'coefficient': lasso_cv.coef_})
lasso_cv_coef['absolute_influence'] = lasso_cv_coef['coefficient'].abs()
lasso_cv_coef = lasso_cv_coef.sort_values('absolute_influence', ascending=False)
display(lasso_cv_coef)
print('Занулені ознаки:', list(lasso_cv_coef.loc[lasso_cv_coef['coefficient'] == 0, 'feature']))


,feature,coefficient,absolute_influence
7,density,-0.646390,0.646390
3,residual sugar,0.525760,0.525760
1,volatile acidity,-0.192033,0.192033
8,pH,0.146200,0.146200
10,alcohol,0.137899,0.137899
0,fixed acidity,0.092847,0.092847
9,sulphates,0.060829,0.060829
5,free sulfur dioxide,0.042751,0.042751
6,total sulfur dioxide,0.012839,0.012839
4,chlorides,0.006822,0.006822


Занулені ознаки: ['citric acid']


**Оцініть середньоквадратичну помилку моделі на навчальній і тестовій вибірках.**

> **Питання 4 : Які середньоквадратичні помилки налаштованої LASSO-регресії на навчальній і відкладеній вибірках?**

In [14]:
print('Середньоквадратична помилка (навчальна вибірка): %.3f' % mean_squared_error(y_train, lasso_cv.predict(X_train_scaled)))
print('Середньоквадратична помилка (відкладена вибірка): %.3f' % mean_squared_error(y_holdout, lasso_cv.predict(X_holdout_scaled)))


Середньоквадратична помилка (навчальна вибірка): 0.558
Середньоквадратична помилка (відкладена вибірка): 0.583


## Випадковий ліс

**Навчіть випадковий ліс з параметрами "з коробки", фіксуючи тільки random_state=17.**

In [15]:
forest = RandomForestRegressor(random_state=17, n_jobs=-1)
forest.fit(X_train, y_train)


RandomForestRegressor(n_jobs=-1, random_state=17)

> **Питання 5 : Які середньоквадратичні помилки випадкового лысу на навчальній вибірці, на крос-валідації (cross_val_score з scoring='neg_mean_squared_error' і іншими параметрами за замовчуванням) і відкладеній вибірках?**

In [16]:
print('Середньоквадратична помилка (навчальна вибірка): %.3f' % mean_squared_error(y_train, forest.predict(X_train)))
print('Середньоквадратична помилка (крос-валідація): %.3f' % (-cross_val_score(forest, X_train, y_train, scoring='neg_mean_squared_error').mean()))
print('Середньоквадратична помилка (відкладена вибірка): %.3f' % mean_squared_error(y_holdout, forest.predict(X_holdout)))


Середньоквадратична помилка (навчальна вибірка): 0.053
Середньоквадратична помилка (крос-валідація): 0.414
Середньоквадратична помилка (відкладена вибірка): 0.371


**Налаштуйте параметри min_samples_leaf і max_depth за допомогою GridSearchCV і знову перевірте якість моделі на крос-валідації і на відкладеній вибірках.**

In [17]:
forest_params = {
    'max_depth': [10, 15, 19],
    'min_samples_leaf': [1, 3, 5],
    'max_features': [7, 10]
}
locally_best_forest = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=17, n_jobs=-1),
    forest_params, scoring='neg_mean_squared_error', cv=3, n_jobs=-1, verbose=1
)
locally_best_forest.fit(X_train, y_train)
print('Найкращі параметри:', locally_best_forest.best_params_)
print('Найкращий MSE на крос-валідації:', -locally_best_forest.best_score_)


Fitting 3 folds for each of 18 candidates, totalling 54 fits
Найкращі параметри: {'max_depth': 19, 'max_features': 7, 'min_samples_leaf': 1}
Найкращий MSE на крос-валідації: 0.4196940462118414


In [18]:
print('Найкращі параметри GridSearchCV:', locally_best_forest.best_params_)
print('MSE на крос-валідації:', -locally_best_forest.best_score_)


Найкращі параметри GridSearchCV: {'max_depth': 19, 'max_features': 7, 'min_samples_leaf': 1}
MSE на крос-валідації: 0.4196940462118414


**Нажал результати GridSearchCV в повному не відтворювані (можуть відрізнятися на різних платформах навіть при фіксованому random_state). Тому навчіть ліс з параметрами max_depth=19, max_features=7, i min_samples_leaf=1 (краще в моэму випадку).**
> **Питання 6 : Які середньоквадратичні помилки налаштованого випадкового лісу на навчальній вибірці, на крос-валідації (cross_val_score з scoring='neg_mean_squared_error') і на відкладеній вибірках?**


In [19]:
tuned_forest = RandomForestRegressor(
    n_estimators=200, max_depth=19, max_features=7,
    min_samples_leaf=1, random_state=17, n_jobs=-1
)
tuned_forest.fit(X_train, y_train)
print('Середньоквадратична помилка (навчальна вибірка): %.3f' % mean_squared_error(y_train, tuned_forest.predict(X_train)))
print('Середньоквадратична помилка (крос-валідація): %.3f' % (-cross_val_score(tuned_forest, X_train, y_train, scoring='neg_mean_squared_error').mean()))
print('Середньоквадратична помилка (відкладена вибірка): %.3f' % mean_squared_error(y_holdout, tuned_forest.predict(X_holdout)))


Середньоквадратична помилка (навчальна вибірка): 0.057
Середньоквадратична помилка (крос-валідація): 0.403
Середньоквадратична помилка (відкладена вибірка): 0.367


**Оцініть важливість ознак за допомогою випадкового лісу.**
>**Питання 7 : Яка ознака виявилася найінформативнішою в налаштованій моделі випадкового лісу?**

In [20]:
rf_importance = pd.DataFrame({'feature': feature_names, 'importance': tuned_forest.feature_importances_})
rf_importance = rf_importance.sort_values('importance', ascending=False)
display(rf_importance)
print('Найінформативніша ознака:', rf_importance.iloc[0]['feature'])


,feature,importance
10,alcohol,0.212744
1,volatile acidity,0.118105
5,free sulfur dioxide,0.111232
7,density,0.088783
3,residual sugar,0.072858
8,pH,0.072329
6,total sulfur dioxide,0.072310
4,chlorides,0.067030
0,fixed acidity,0.063033
2,citric acid,0.062305


Найінформативніша ознака: alcohol


**Висновок**

У роботі було порівняно лінійну регресію, Lasso-регресію та випадковий ліс для прогнозування якості білого вина.

Лінійна регресія є базовою моделлю та дає змогу оцінити напрям і силу впливу ознак за допомогою коефіцієнтів. Lasso додатково виконує відбір ознак: регуляризація зменшує коефіцієнти, а частину з них може занулити.

Випадковий ліс враховує нелінійні залежності між характеристиками вина та якістю. Його важливість ознак не має знака, тому показує силу впливу, але не напрям. Налаштування параметрів за допомогою GridSearchCV дає змогу зменшити помилку на відкладеній вибірці порівняно з базовою моделлю.

Для остаточного порівняння потрібно орієнтуватися на MSE відкладеної вибірки: чим менше це значення, тим точніше модель прогнозує якість вина. Різниця між MSE на навчальній і відкладеній вибірках свідчить про рівень перенавчання моделі.